In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, warnings, zipfile, subprocess, importlib.util
ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME']=str(WORK/'.cache'/'huggingface')
os.environ['TORCH_HOME']=str(WORK/'.cache'/'torch')
os.environ['MPLCONFIGDIR']=str(WORK/'.cache'/'matplotlib')
CFG = {
    'seed': 2026, 'folds': 5, 'max_hours': 8.0, 'batch_size': 32,
    'backbones': ['vit_small_patch14_dinov2.lvd142m', 'vit_base_patch14_dinov2.lvd142m'],
    'image_size': 224, 'slices_per_series': 24, 'max_series_per_plane': 2,
    'device': 'cuda', 'allow_downloads': True,
    # Set from the official evaluation page, or attach knee_metric.json.
    # Supported: macro_auc, macro_log_loss, weighted_log_loss, custom.
    'metric': 'auto', 'label_weights': {}, 'custom_metric_path': None,
    'labels_csv': None, 'sample_csv': None, 'id_column': None,
    'patient_column': None, 'label_columns': None,
    'dicom_root': None, 'series_csv': None,
    # Explicit test row mapping for long-format templates: CSV with row_id,study_id,label.
    'submission_map_csv': None,
    # Optional trusted external predictions: see the artifact audit cell.
    'external_manifests': [],
}
START = time.monotonic()
def atomic_json(path, obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str)); tmp.replace(path)
def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()
def remaining(): return CFG['max_hours']*3600-(time.monotonic()-START)
atomic_json(WORK/'requested_config.json',CFG)


In [ ]:
required={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn',
          'torch':'torch','torchvision':'torchvision','timm':'timm',
          'pydicom':'pydicom','PIL':'Pillow','scipy':'scipy'}
missing=[v for k,v in required.items() if importlib.util.find_spec(k) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
import numpy as np, pandas as pd
import torch, timm, pydicom
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from scipy.special import expit, logit
from torch.utils.data import Dataset, DataLoader
from timm.data import resolve_model_data_config, create_transform
np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
torch.set_num_threads(min(4,os.cpu_count() or 1))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,'; installed missing packages:',missing)
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(WORK/'environment.txt','w'),check=True)


In [ ]:
# Inventory is metadata-only. Do not unzip opaque model files or load pickled experiments.
FILES=sorted(p for p in ROOT.rglob('*') if p.is_file()) if ROOT.exists() else []
BACKUP=WORK/'recovered_artifacts'; BACKUP.mkdir(exist_ok=True)
for zpath in [p for p in FILES if p.name=='knee_experiment_backup.zip']:
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            relative=Path(info.filename)
            if relative.is_absolute() or '..' in relative.parts: continue
            if info.file_size>2_000_000_000: continue
            if relative.suffix.lower() not in {'.csv','.json','.npz','.npy','.safetensors','.pth','.pt'}: continue
            target=BACKUP/digest(str(zpath))[:10]/relative
            target.parent.mkdir(parents=True,exist_ok=True)
            if not target.exists():
                with z.open(info) as src, target.open('wb') as dst:
                    import shutil
                    shutil.copyfileobj(src,dst)
FILES+=sorted(p for p in BACKUP.rglob('*') if p.is_file())
CSV={}
notebook_audit=[]
for p in FILES:
    if p.suffix=='.ipynb':
        try:
            nb=json.loads(p.read_text())
            source='\n'.join(''.join(c.get('source',[])) for c in nb.get('cells',[]) if c.get('cell_type')=='code')
            notebook_audit.append({'path':str(p),'detected_terms':[term for term in ['dinov2','resnet','roc_auc_score','log_loss','GroupKFold','StratifiedGroupKFold'] if term.lower() in source.lower()]})
        except Exception: pass
atomic_json(WORK/'prior_notebook_audit.json',notebook_audit)
for p in FILES:
    if p.suffix.lower()=='.csv':
        try: CSV[str(p)]=pd.read_csv(p,nrows=5)
        except Exception: pass
atomic_json(WORK/'input_inventory.json',{
    'csv_schemas':{p:list(df.columns) for p,df in CSV.items()},
    'notebooks':[str(p) for p in FILES if p.suffix=='.ipynb'],
    'artifacts':[str(p) for p in FILES if p.suffix in {'.npz','.npy','.pt','.pth','.safetensors'}],
})
for p in FILES:
    if p.name=='knee_metric.json':
        metric_cfg=json.loads(p.read_text())
        for k in ['metric','label_weights','custom_metric_path']:
            if k in metric_cfg: CFG[k]=metric_cfg[k]
print('Files:',len(FILES),'CSV files:',len(CSV),'prior notebooks:',sum(p.suffix=='.ipynb' for p in FILES))
for p,df in list(CSV.items())[:20]: print(Path(p).name, list(df.columns)[:16])


In [ ]:
def choose_csv(override, names, purpose):
    if override: return Path(override)
    candidates=[Path(p) for p in CSV if Path(p).name.lower() in names]
    if 'SAMPLE_PATH' in globals():
        colocated=[p for p in candidates if p.parent==SAMPLE_PATH.parent]
        if len(colocated)==1: candidates=colocated
    if len(candidates)!=1:
        raise ValueError(f'Cannot uniquely identify {purpose}: {candidates}. Set the corresponding CFG path.')
    return candidates[0]
SAMPLE_PATH=choose_csv(CFG['sample_csv'],{'sample_submission.csv'},'sample submission')
LABEL_PATH=choose_csv(CFG['labels_csv'],{'train.csv','train_labels.csv','labels.csv'},'official labels')
sample=pd.read_csv(SAMPLE_PATH,dtype=str)
train=pd.read_csv(LABEL_PATH,dtype=str)
id_candidates=[c for c in ['study_id','StudyInstanceUID','StudyInstanceUid','study_uid'] if c in train]
ID=CFG['id_column'] or (id_candidates[0] if len(id_candidates)==1 else None)
if ID is None: raise ValueError('Set CFG[id_column] to the study identifier in the official labels.')
if train[ID].isna().any() or train[ID].duplicated().any():
    raise ValueError('Official labels must contain exactly one row per study. Long labels need an explicit schema adapter.')
LABELS=CFG['label_columns']
if LABELS is None:
    LABELS=[]
    excluded={ID,CFG['patient_column'],'patient_id','fold','split'}
    for c in train:
        if c in excluded: continue
        v=pd.to_numeric(train[c],errors='coerce')
        if v.notna().any() and v.dropna().isin([0,1]).all(): LABELS.append(c)
if len(LABELS)!=12:
    raise ValueError(f'Found {len(LABELS)} binary label columns: {LABELS}. Set CFG[label_columns] from official documentation.')
y=train[LABELS].apply(pd.to_numeric,errors='raise').to_numpy(float)
if not np.isin(y[~np.isnan(y)],[0,1]).all(): raise ValueError('Labels must be binary or missing.')
# Only official complete-label studies enter model selection; reports never become features.
complete=np.isfinite(y).all(1)
print('Official studies:',len(train),'complete:',int(complete.sum()),'; labels:',LABELS)
train=train.loc[complete].reset_index(drop=True); y=y[complete]
TRAIN_IDS=train[ID].astype(str).tolist()
if len(TRAIN_IDS)<10: raise ValueError('Too few completely labelled studies for this workflow.')
PATIENT=CFG['patient_column'] or ('patient_id' if 'patient_id' in train else ID)
groups=train[PATIENT].fillna(train[ID]).astype(str).to_numpy()
if PATIENT==ID: print('Patient IDs unavailable: folds are study-disjoint; cross-study patient identity cannot be checked.')
# Exact sample order is retained; unknown row formats are never guessed by string splitting.
if ID in sample and set(LABELS).issubset(sample.columns):
    MODE='wide'; TEST_IDS=sample[ID].astype(str).drop_duplicates().tolist(); row_map=None
elif CFG['submission_map_csv']:
    MODE='long'; row_map=pd.read_csv(CFG['submission_map_csv'],dtype=str)
    if not {'row_id','study_id','label'}.issubset(row_map): raise ValueError('Invalid explicit row map.')
    if 'row_id' not in sample or not row_map.row_id.equals(sample.row_id): raise ValueError('Row map must match sample row order.')
    if not set(row_map.label).issubset(LABELS): raise ValueError('Unknown label in row map.')
    TEST_IDS=row_map.study_id.drop_duplicates().tolist()
else:
    raise ValueError('Unrecognized submission schema. Set submission_map_csv; do not guess study IDs from row strings.')
if set(TRAIN_IDS)&set(TEST_IDS): raise ValueError('Training/test study overlap; inspect the selected CSVs.')
ALL_IDS=TRAIN_IDS+TEST_IDS
DATA_SIGNATURE=digest({'ids':TRAIN_IDS,'labels':LABELS,'y':y.tolist(),'patients':groups.tolist()})


In [ ]:
# Reuse folds only when IDs and group isolation are verifiable.
fold_file=WORK/'knee_5fold_assignments.csv'
fold_candidates=[fold_file] if fold_file.exists() else [p for p in FILES if p.name=='knee_5fold_assignments.csv']
folds=None
for p in fold_candidates:
    d=pd.read_csv(p,dtype={ID:str})
    if ID not in d or 'fold' not in d or d[ID].duplicated().any(): continue
    if not set(TRAIN_IDS).issubset(set(d[ID])): continue
    f=pd.to_numeric(d.set_index(ID).loc[TRAIN_IDS,'fold'],errors='coerce').to_numpy()
    if not np.isfinite(f).all() or not np.equal(f,np.floor(f)).all(): continue
    if len(np.unique(f))!=CFG['folds']: continue
    if pd.DataFrame({'g':groups,'f':f}).groupby('g').f.nunique().max()!=1: continue
    folds=pd.factorize(f,sort=True)[0]; break
if folds is None:
    # Search deterministic group folds for balanced, usable binary validation targets.
    best=None
    for seed in range(CFG['seed'],CFG['seed']+128):
        splitter=StratifiedGroupKFold(CFG['folds'],shuffle=True,random_state=seed)
        strata=np.argmax(y/(y.mean(0)+1e-3),axis=1)
        f=np.full(len(y),-1)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for k,(_,va) in enumerate(splitter.split(y,strata,groups)): f[va]=k
        if (f<0).any(): continue
        bad=sum(int(np.unique(y[f==k,j]).size<2) for k in range(CFG['folds']) for j in range(len(LABELS)))
        balance=sum(np.square(y[f==k].mean(0)-y.mean(0)).sum() for k in range(CFG['folds']))
        candidate=(bad+balance,f)
        if best is None or candidate[0]<best[0]: best=candidate
    if best is None: raise ValueError('Unable to construct folds.')
    folds=best[1]
assert pd.DataFrame({'g':groups,'f':folds}).groupby('g').f.nunique().max()==1
fold_df=pd.DataFrame({ID:TRAIN_IDS,'patient_group':groups,'fold':folds})
fold_df.to_csv(fold_file,index=False)
FOLD_SIGNATURE=digest({'data':DATA_SIGNATURE,'folds':folds.tolist()})
print('Fold sizes:',np.bincount(folds).tolist())
# The metric must be explicitly verified from official competition documentation.
if CFG['metric'] in {None,'auto'}:
    # The API evaluationMetric field is official metadata, not guessed from prior notebooks.
    import urllib.request, urllib.parse, re
    metric_evidence=[]; official_names=[]
    url='https://www.kaggle.com/api/v1/competitions/list?search=rsna-knee-abnormality-detection'
    try:
        with urllib.request.urlopen(url,timeout=25) as response: payload=json.load(response)
        records=payload if isinstance(payload,list) else payload.get('competitions',[])
        for record in records:
            ref=str(record.get('ref',record.get('competitionName',''))).rstrip('/').split('/')[-1]
            if ref=='rsna-knee-abnormality-detection':
                metric=record.get('evaluationMetric',record.get('evaluation_metric'))
                metric_evidence.append(record)
                if metric: official_names.append(str(metric).strip().lower())
    except Exception as exc:
        metric_evidence.append({'public_api_error':type(exc).__name__})
    if not official_names and importlib.util.find_spec('kaggle'):
        # Reuse injected credentials only when present; never prompt or print their values.
        has_binding=bool(os.environ.get('KAGGLE_API_TOKEN') or
            (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')) or
            (Path.home()/'.kaggle'/'kaggle.json').exists())
        if has_binding:
            try:
                from kaggle.api.kaggle_api_extended import KaggleApi
                api=KaggleApi(); api.authenticate()
                for record in api.competitions_list(search='rsna-knee-abnormality-detection'):
                    ref=str(getattr(record,'ref','')).rstrip('/').split('/')[-1]
                    if ref=='rsna-knee-abnormality-detection':
                        name=getattr(record,'evaluation_metric',None) or getattr(record,'evaluationMetric',None)
                        if name: official_names.append(str(name).strip().lower())
            except Exception as exc: metric_evidence.append({'authenticated_api_error':type(exc).__name__})
    atomic_json(WORK/'official_metric_discovery.json',{'names':official_names,'evidence':metric_evidence})
    # Only unambiguous macro metrics are automatic. Weighted or task-specific scores
    # require their actual weights/custom scorer, never an assumed equal-weight proxy.
    mapping={'mean columnwise roc auc':'macro_auc','mean column-wise roc auc':'macro_auc',
             'macro averaged roc auc':'macro_auc','macro-averaged roc auc':'macro_auc',
             'mean columnwise log loss':'macro_log_loss','mean column-wise log loss':'macro_log_loss'}
    mapped={mapping[name] for name in official_names if name in mapping}
    if len(mapped)==1 and all(name in mapping for name in official_names): CFG['metric']=mapped.pop()
    else:
        raise ValueError('Could not verify the exact 12-label metric automatically. See official_metric_discovery.json. Set CFG[metric] and official weights/custom scorer from the evaluation page, or attach knee_metric.json. ROC AUC alone does not specify multilabel aggregation.')
if CFG['metric'] not in {'macro_auc','macro_log_loss','weighted_log_loss','custom'}: raise ValueError('Unsupported metric.')
weights=np.array([CFG['label_weights'].get(c,1.) for c in LABELS],float)
if CFG['metric']=='weighted_log_loss' and set(CFG['label_weights'])!=set(LABELS):
    raise ValueError('Supply every official label weight explicitly.')
if np.any(weights<0) or weights.sum()<=0: raise ValueError('Invalid metric weights.')
CUSTOM=None
if CFG['metric']=='custom':
    import runpy
    CUSTOM=runpy.run_path(CFG['custom_metric_path'])['score']
    # Function contract: score(y_true, probabilities, label_names) -> scalar; larger is better.
def per_label(yt,p):
    p=np.clip(p,1e-6,1-1e-6)
    return {c:{'auc':float(roc_auc_score(yt[:,j],p[:,j])) if len(np.unique(yt[:,j]))==2 else None,
               'log_loss':float(log_loss(yt[:,j],p[:,j],labels=[0,1]))} for j,c in enumerate(LABELS)}
def score(yt,p):
    if not np.isfinite(p).all() or p.shape!=yt.shape: raise ValueError('Invalid prediction matrix.')
    if CUSTOM: return float(CUSTOM(yt,p,LABELS))
    m=per_label(yt,p)
    if CFG['metric']=='macro_auc':
        if any(m[c]['auc'] is None for c in LABELS): raise ValueError('AUC undefined: a label has only one class in this evaluation partition.')
        return float(np.mean([m[c]['auc'] for c in LABELS]))
    v=np.array([m[c]['log_loss'] for c in LABELS])
    return -float(np.average(v,weights=weights if CFG['metric']=='weighted_log_loss' else None))
atomic_json(WORK/'resolved_config.json',dict(CFG,labels=LABELS,data_signature=DATA_SIGNATURE,fold_signature=FOLD_SIGNATURE))


In [ ]:
# Index headers once. No reports, labels, or acquisition dates enter the image representation.
INDEX_PATH=WORK/'mri_index.json'
source_signature=digest([(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in FILES if p.suffix.lower() in {'.dcm','.dicom','.npy'}])
index_payload=json.loads(INDEX_PATH.read_text()) if INDEX_PATH.exists() else {}
if index_payload.get('source_signature')==source_signature:
    index=index_payload['index']
else:
    index={s:{} for s in ALL_IDS}; errors=0
    for p in FILES:
        if CFG['dicom_root'] and not p.is_relative_to(Path(CFG['dicom_root'])): continue
        if p.suffix.lower() not in {'.dcm','.dicom'}: continue
        try:
            ds=pydicom.dcmread(p,stop_before_pixels=True)
            sid=str(ds.get('StudyInstanceUID',''))
            # Competition IDs often appear as directory names rather than DICOM UIDs.
            matches=[part for part in p.parts if part in index]
            if sid not in index: sid=matches[-1] if len(matches)==1 else None
            if sid is None: continue
            series=str(ds.get('SeriesInstanceUID',p.parent.name))
            orient=np.asarray(ds.get('ImageOrientationPatient',[]),float)
            normal=np.cross(orient[:3],orient[3:]) if orient.size==6 else None
            if normal is not None:
                axis=int(np.abs(normal).argmax()); plane=['sagittal','coronal','axial'][axis]
                position=float(np.dot(np.asarray(ds.get('ImagePositionPatient',[0,0,0]),float),normal))
            else:
                desc=str(ds.get('SeriesDescription','')).lower()
                plane=next((v for v in ['sagittal','coronal','axial'] if v[:3] in desc),'unknown')
                position=float(ds.get('InstanceNumber',0))
            entry=index[sid].setdefault(series,{'plane':plane,'slices':[]})
            entry['slices'].append([position,str(p)])
        except Exception: errors+=1
    for study in index.values():
        for ser in study.values(): ser['slices'].sort(key=lambda item:(item[0],item[1]))
    atomic_json(INDEX_PATH,{'source_signature':source_signature,'index':index,'header_errors':errors})
missing_studies=[s for s in ALL_IDS if not index.get(s)]
print('DICOM-indexed studies:',len(ALL_IDS)-len(missing_studies),'missing:',len(missing_studies))
# Compatible feature manifests may supply missing DICOMs; checked by the next cell.


In [ ]:
# Cache contract: feature_manifest.json beside an NPZ with ids and features (N,4,3,D).
# Slots are sagittal/coronal/axial/unknown; pools are center-mean, all-mean, all-std.
# Frozen unsupervised encoders only; supervised caches need fold-specific extraction and are rejected.
PLANES=['sagittal','coronal','axial','unknown']
PREPROCESS={'percentiles':[0.5,99.5],'image_size':CFG['image_size'],
            'slice_count':CFG['slices_per_series'],'max_series':CFG['max_series_per_plane'],
            'version':3,'plane_order':PLANES,'pools':['center_mean','all_mean','all_std']}
def load_feature_cache(manifest,backbone):
    try:
        m=json.loads(Path(manifest).read_text())
        if m.get('backbone')!=backbone or m.get('preprocess')!=PREPROCESS: return None
        if m.get('encoder_training')!='frozen_public_pretrained': return None
        p=Path(manifest).parent/m['file']
        if m.get('sha256')!=hashlib.sha256(p.read_bytes()).hexdigest(): return None
        z=np.load(p,allow_pickle=False); ids=z['ids'].astype(str).tolist(); a=z['features']
        if len(ids)!=len(set(ids)) or a.shape[:3]!=(len(ids),4,3) or not np.isfinite(a).all(): return None
        if not set(ALL_IDS).issubset(ids): return None
        pos={s:i for i,s in enumerate(ids)}
        return a[[pos[s] for s in ALL_IDS]]
    except Exception: return None
def normalize_pixels(ds):
    arr=ds.pixel_array.astype(np.float32)
    if arr.ndim!=2: raise ValueError('Only single-frame grayscale DICOMs are supported; multi-frame needs an explicit adapter.')
    arr=arr*float(ds.get('RescaleSlope',1))+float(ds.get('RescaleIntercept',0))
    valid=np.isfinite(arr)
    if 'PixelPaddingValue' in ds: valid &= arr!=float(ds.PixelPaddingValue)*float(ds.get('RescaleSlope',1))+float(ds.get('RescaleIntercept',0))
    if not valid.any(): raise ValueError('Empty MRI image.')
    lo,hi=np.percentile(arr[valid],PREPROCESS['percentiles'])
    arr=np.nan_to_num(np.clip((arr-lo)/max(float(hi-lo),1e-6),0,1))
    if str(ds.get('PhotometricInterpretation',''))=='MONOCHROME1': arr=1-arr
    return Image.fromarray(np.uint8(arr*255)).convert('RGB')
class SliceDataset(Dataset):
    def __init__(self,paths,transform): self.paths=paths; self.transform=transform
    def __len__(self): return len(self.paths)
    def __getitem__(self,i): return self.transform(normalize_pixels(pydicom.dcmread(self.paths[i])))
def features_for(backbone):
    key=digest({'backbone':backbone,'preprocess':PREPROCESS})[:16]
    folder=WORK/'features'/key; folder.mkdir(parents=True,exist_ok=True)
    manifests=[folder/'feature_manifest.json']+[p for p in FILES if p.name=='feature_manifest.json']
    for manifest in manifests:
        a=load_feature_cache(manifest,backbone)
        if a is not None:
            print('Reused verified features:',backbone); return a
    if missing_studies: raise ValueError(f'Missing MRI images for {len(missing_studies)} studies and no compatible complete feature cache. Examples: {missing_studies[:4]}')
    # Study shards allow resuming interrupted feature extraction without recomputing completed studies.
    checkpoint=None
    for p in FILES:
        if p.name=='encoder_manifest.json':
            m=json.loads(p.read_text())
            if m.get('backbone')==backbone and m.get('encoder_training')=='frozen_public_pretrained':
                candidate=p.parent/m['file']
                if hashlib.sha256(candidate.read_bytes()).hexdigest()==m.get('sha256'): checkpoint=candidate; break
    if checkpoint is None and not CFG['allow_downloads']: raise RuntimeError('Attach a verified pretrained encoder_manifest.json or enable downloads.')
    model=timm.create_model(backbone,pretrained=checkpoint is None,num_classes=0,img_size=CFG['image_size'])
    if checkpoint:
        if checkpoint.suffix=='.safetensors':
            from safetensors.torch import load_file
            state=load_file(str(checkpoint))
        else: state=torch.load(checkpoint,map_location='cpu',weights_only=True)
        model.load_state_dict(state,strict=True)
    model=model.eval().to(DEVICE)
    transform=create_transform(**dict(resolve_model_data_config(model),input_size=(3,CFG['image_size'],CFG['image_size'])),is_training=False)
    # Use encoder's documented resolution; interpolation is deterministic.
    dim=model.num_features; result=[]
    for si,sid in enumerate(ALL_IDS):
        shard=folder/(digest(sid)+'.npz')
        if shard.exists():
            z=np.load(shard,allow_pickle=False)
            if str(z['study_id'])==sid and str(z['source'])==source_signature:
                a=z['features']
                if a.shape==(4,3,dim) and np.isfinite(a).all(): result.append(a); continue
        if remaining()<120: raise TimeoutError('Time budget reached. Completed feature shards are saved; resume in a fresh session.')
        a=np.zeros((4,3,dim),np.float32)
        for pi,plane in enumerate(PLANES):
            series=[s for s in index[sid].values() if s['plane']==plane]
            series=sorted(series,key=lambda s:(-len(s['slices']),s['slices'][0][1]))[:CFG['max_series_per_plane']]
            pooled=[]
            for ser in series:
                paths=[v[1] for v in ser['slices']]
                sel=np.unique(np.rint(np.linspace(0,len(paths)-1,min(len(paths),CFG['slices_per_series']))).astype(int))
                paths=[paths[i] for i in sel]
                loader=DataLoader(SliceDataset(paths,transform),batch_size=CFG['batch_size'],shuffle=False,num_workers=0)
                embeddings=[]
                with torch.inference_mode():
                    for batch in loader:
                        with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                            out=model(batch.to(DEVICE))
                        embeddings.append(out.float().cpu().numpy())
                e=np.concatenate(embeddings); n=len(e); mid=e[max(0,n//4):max(1,n-n//4)]
                pooled.append(np.stack([mid.mean(0),e.mean(0),e.std(0)]))
            if pooled: a[pi]=np.mean(pooled,axis=0)
        temp=shard.with_suffix('.tmp.npz'); np.savez_compressed(temp,features=a,study_id=sid,source=source_signature); temp.replace(shard)
        result.append(a)
        if (si+1)%10==0: print(backbone,si+1,'/',len(ALL_IDS),'studies cached')
    a=np.stack(result); target=folder/'features.npz'
    np.savez_compressed(target,ids=np.asarray(ALL_IDS),features=a)
    atomic_json(folder/'feature_manifest.json',{'backbone':backbone,'preprocess':PREPROCESS,'encoder_training':'frozen_public_pretrained',
        'file':target.name,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()})
    del model
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return a


In [ ]:
# Small-data heads: regularization and pooling are selected inside each outer fold.
# Every scaler is fit only on its training partition. No supervised full-data feature extraction.
def representation(a,mode):
    if mode=='center': return a[:,:,0,:].reshape(len(a),-1)
    if mode=='all': return a[:,:,1,:].reshape(len(a),-1)
    if mode=='mean_std': return a[:,:,1:,:].reshape(len(a),-1)
    raise ValueError(mode)
def fit_head(x,yt,C):
    models=[]
    for j in range(yt.shape[1]):
        if len(np.unique(yt[:,j]))<2: models.append(float((yt[:,j].sum()+1)/(len(yt)+2))); continue
        model=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=1500,solver='liblinear',random_state=CFG['seed']))
        model.fit(x,yt[:,j]); models.append(model)
    return models
def predict_head(models,x):
    return np.column_stack([np.full(len(x),m) if isinstance(m,float) else m.predict_proba(x)[:,1] for m in models])
def inner_splits(indices):
    n=min(3,len(np.unique(groups[indices])))
    if n<2: raise ValueError('Too few groups for inner validation.')
    return [(indices[a],indices[b]) for a,b in GroupKFold(n).split(indices,groups=groups[indices])]
def tune(a,indices):
    candidates=[]
    for mode in ['center','all','mean_std']:
        x=representation(a,mode)
        for C in [0.001,0.01,0.1,1.0]:
            p=np.zeros((len(indices),len(LABELS))); positions={v:i for i,v in enumerate(indices)}
            for tr,va in inner_splits(indices):
                pred=predict_head(fit_head(x[tr],y[tr],C),x[va])
                p[[positions[v] for v in va]]=pred
            candidates.append((score(y[indices],p),mode,C))
    return sorted(candidates,key=lambda v:v[0],reverse=True)[0],candidates
EXPERIMENTS={}; n=len(y)
for backbone in CFG['backbones']:
    if remaining()<300: print('Skipping optional backbone due to time budget:',backbone); continue
    try:
        a=features_for(backbone)
    except (TimeoutError, RuntimeError, OSError) as exc:
        if not EXPERIMENTS: raise
        print('Optional backbone incomplete:',backbone,type(exc).__name__,'; retaining completed models.')
        atomic_json(WORK/'optional_backbone_failure.json',{'backbone':backbone,'error_type':type(exc).__name__,'message':str(exc)[:500]})
        continue
    key=digest({'backbone':backbone,'fold':FOLD_SIGNATURE,'metric':CFG['metric'],'weights':weights.tolist(),'preprocess':PREPROCESS,'test_ids':TEST_IDS,'feature_hash':hashlib.sha256(a.tobytes()).hexdigest(),'version':4})[:16]
    folder=WORK/'experiments'/key; folder.mkdir(parents=True,exist_ok=True)
    oof=np.zeros_like(y); test_fold=[]; settings=[]
    for k in range(CFG['folds']):
        tr=np.flatnonzero(folds!=k); va=np.flatnonzero(folds==k); saved=folder/f'fold_{k}.npz'; meta=folder/f'fold_{k}.json'
        if saved.exists() and meta.exists():
            z=np.load(saved,allow_pickle=False)
            if np.array_equal(z['valid_indices'],va):
                oof[va]=z['valid']; test_fold.append(z['test']); settings.append(json.loads(meta.read_text())); continue
        best,trials=tune(a,tr); _,mode,C=best
        x=representation(a,mode); models=fit_head(x[tr],y[tr],C)
        pv=predict_head(models,x[va]); pt=predict_head(models,x[n:])
        np.savez_compressed(saved,valid_indices=va,valid=pv,test=pt)
        detail={'fold':k,'pooling':mode,'C':C,'inner_score':best[0],'trials':trials}
        atomic_json(meta,detail); settings.append(detail)
        # Joblib is written for resume/export only; never load untrusted attached pickles.
        import joblib
        joblib.dump(models,folder/f'head_fold_{k}.joblib')
        oof[va]=pv; test_fold.append(pt)
        print(backbone,'fold',k,'pool',mode,'C',C)
    test=np.mean(test_fold,axis=0)
    EXPERIMENTS[backbone]={'oof':oof,'test':test,'test_folds':np.stack(test_fold),'settings':settings}
    frame=pd.DataFrame(oof,columns=LABELS); frame.insert(0,ID,TRAIN_IDS); frame['fold']=folds
    frame.to_csv(folder/'oof.csv',index=False)
    tf=pd.DataFrame(test,columns=LABELS); tf.insert(0,ID,TEST_IDS); tf.to_csv(folder/'test.csv',index=False)
    atomic_json(folder/'metrics.json',{'score_higher_is_better':score(y,oof),'per_label':per_label(y,oof),
        'fold_signature':FOLD_SIGNATURE,'backbone':backbone})
    print(backbone,'nested OOF score:',round(score(y,oof),6))
if not EXPERIMENTS: raise RuntimeError('No complete model experiment. Resume cached extraction with a larger time budget.')


In [ ]:
# Audit prior ResNet/all-slices/OOF artifacts. CSVs alone do not prove leakage-free provenance.
# Trusted manifest schema:
# {name, oof_csv, test_csv, id_column, label_columns, fold_signature,
#  data_signature, validation_protocol:'outer_fold_independent',
#  supervised_training_excludes_validation_studies:true,
#  preprocessing_fit_on_training_only:true, public_test_used_for_selection:false}
# Hyperparameters must be fixed independently of the outer labels. Prior OOF tuned on all
# outer labels is useful for descriptive comparison only, not an unbiased model candidate.
audit=[]
for p in FILES:
    if p.suffix=='.csv' and any(s in p.name.lower() for s in ['oof','resnet','submission','prediction']):
        audit.append({'path':str(p),'status':'unverified; excluded from selection'})
for path in list(dict.fromkeys(CFG['external_manifests']+[str(p) for p in FILES if p.name.endswith('prediction_manifest.json')])):
    path=Path(path); m=json.loads(path.read_text())
    required=(m.get('fold_signature')==FOLD_SIGNATURE and m.get('data_signature')==DATA_SIGNATURE
        and m.get('validation_protocol')=='outer_fold_independent'
        and m.get('supervised_training_excludes_validation_studies') is True
        and m.get('preprocessing_fit_on_training_only') is True
        and m.get('public_test_used_for_selection') is False)
    if not required: audit.append({'path':str(path),'status':'provenance mismatch; excluded'}); continue
    mid=m.get('id_column',ID)
    def aligned(file,ids):
        df=pd.read_csv(path.parent/file,dtype={mid:str})
        if df[mid].duplicated().any() or not set(ids).issubset(df[mid]): raise ValueError('Prediction IDs missing or duplicated.')
        v=df.set_index(mid).loc[ids,LABELS].to_numpy(float)
        if not np.isfinite(v).all() or np.any((v<0)|(v>1)): raise ValueError('Invalid external probabilities.')
        return v
    oo=aligned(m['oof_csv'],TRAIN_IDS); tt=aligned(m['test_csv'],TEST_IDS)
    # Descriptive only: external OOF can share trained folds with the outer meta-validation.
    # Safe ensemble evaluation would require nested external predictions, not ordinary OOF.
    audit.append({'path':str(path),'status':'aligned; descriptive comparison only',
                  'score':score(y,oo),'per_label':per_label(y,oo)})
atomic_json(WORK/'artifact_audit.json',audit)
print('Prior prediction files audited:',len(audit),'; details: artifact_audit.json')


In [ ]:
# Nested selection of backbone and ensemble weights: inner predictions are regenerated using
# only outer-training studies. Outer validation labels never select that fold's ensemble.
# This deliberately costs extra small CPU heads, while reusing all frozen MRI features.
names=list(EXPERIMENTS)
A={name:features_for(name) for name in names}
ensemble_oof=np.zeros_like(y); ensemble_tests=[]; decisions=[]
for k in range(CFG['folds']):
    tr=np.flatnonzero(folds!=k); va=np.flatnonzero(folds==k)
    inner_predictions=[]; outer_predictions=[]; outer_tests=[]
    for name in names:
        a=A[name]; p=np.zeros((len(tr),len(LABELS))); pos={v:i for i,v in enumerate(tr)}
        # Reuse the outer-training-selected configuration. Inner model selection remains
        # optimistic locally, but outer validation remains fully untouched.
        setting=EXPERIMENTS[name]['settings'][k]; x=representation(a,setting['pooling'])
        for it,iv in inner_splits(tr):
            p[[pos[v] for v in iv]]=predict_head(fit_head(x[it],y[it],setting['C']),x[iv])
        inner_predictions.append(p)
        outer_predictions.append(EXPERIMENTS[name]['oof'][va])
        outer_tests.append(EXPERIMENTS[name]['test_folds'][k])
    if len(names)==1: candidates=[np.ones(1)]
    elif len(names)==2: candidates=[np.array([w,1-w]) for w in [0.,0.25,0.5,0.75,1.]]
    else:
        candidates=[np.eye(len(names))[i] for i in range(len(names))]+[np.ones(len(names))/len(names)]
    # Small predefined weight grid limits tuning noise on 58 studies.
    ws=max(candidates,key=lambda w:score(y[tr],np.tensordot(w,np.stack(inner_predictions),axes=1)))
    ensemble_oof[va]=np.tensordot(ws,np.stack(outer_predictions),axes=1)
    ensemble_tests.append(np.tensordot(ws,np.stack(outer_tests),axes=1))
    decisions.append({'fold':k,'weights':dict(zip(names,ws.tolist()))})
final_test=np.mean(ensemble_tests,axis=0)
metrics={'metric':CFG['metric'],'score_direction':'higher is better; log losses are negated',
         'nested_ensemble_score':score(y,ensemble_oof),'per_label':per_label(y,ensemble_oof),
         'decisions':decisions,'n_studies':len(y),
         'warning':'58 labelled studies imply high uncertainty; no leaderboard target is guaranteed.'}
# Group bootstrap reflects study/patient sampling uncertainty; undefined AUC samples are skipped.
rng=np.random.default_rng(CFG['seed']); boot=[]; unique=np.unique(groups)
for _ in range(500):
    chosen=rng.choice(unique,len(unique),replace=True)
    ix=np.concatenate([np.flatnonzero(groups==g) for g in chosen])
    try: boot.append(score(y[ix],ensemble_oof[ix]))
    except ValueError: pass
metrics['bootstrap_valid_replicates']=len(boot)
metrics['bootstrap_95_interval']=np.quantile(boot,[.025,.975]).tolist() if len(boot)>=50 else None
atomic_json(WORK/'validation_metrics.json',metrics)
out=pd.DataFrame(ensemble_oof,columns=LABELS); out.insert(0,ID,TRAIN_IDS); out['fold']=folds
out.to_csv(WORK/'knee_5fold_oof_predictions.csv',index=False)
print('Nested ensemble:',round(metrics['nested_ensemble_score'],6),'bootstrap interval:',metrics['bootstrap_95_interval'])
print(pd.DataFrame(metrics['per_label']).T.round(5).to_string())


In [ ]:
# Build the exact template without reordering or inventing predictions.
if final_test.shape!=(len(TEST_IDS),len(LABELS)): raise ValueError('Wrong test prediction shape.')
if not np.isfinite(final_test).all(): raise ValueError('Nonfinite final predictions.')
final_test=np.clip(final_test,1e-6,1-1e-6)
submission=sample.copy()
if MODE=='wide':
    lookup=pd.DataFrame(final_test,index=TEST_IDS,columns=LABELS)
    submission[LABELS]=lookup.loc[sample[ID].astype(str),LABELS].to_numpy()
    pred_cols=LABELS
else:
    pred_cols=[c for c in sample if c!='row_id']
    if len(pred_cols)!=1: raise ValueError('Long binary schema must have exactly one probability column.')
    si={s:i for i,s in enumerate(TEST_IDS)}; li={c:i for i,c in enumerate(LABELS)}
    submission[pred_cols[0]]=[final_test[si[s],li[l]] for s,l in zip(row_map.study_id,row_map.label)]
assert list(submission)==list(sample) and len(submission)==len(sample)
for c in sample:
    if c not in pred_cols: assert submission[c].equals(sample[c])
pred=submission[pred_cols].to_numpy(float)
assert np.isfinite(pred).all() and ((pred>=0)&(pred<=1)).all()
path=WORK/'submission.csv'; temp=WORK/'submission.tmp.csv'
submission.to_csv(temp,index=False); temp.replace(path)
assert pd.read_csv(path).shape==sample.shape
# Save enough state for restart. Kaggle /working must be committed/downloaded before
# a session is destroyed; a fresh kernel can attach that output as an input dataset.
atomic_json(WORK/'run_complete.json',{'submission':str(path),'rows':len(submission),
    'columns':list(submission),'data_signature':DATA_SIGNATURE,'fold_signature':FOLD_SIGNATURE,
    'elapsed_minutes':(time.monotonic()-START)/60})
from IPython.display import display, FileLink
display(FileLink(str(path)))
print('Saved:',path,'; commit notebook output to retain caches across sessions.')
